# PNKLN Core Stack™: Multi-Agent Coordination Template
**Version:** 1.0.0  
**Target:** Vertex AI Workbench (Google Cloud)  
**Governance:** AiYouJR + ATP 5-19 + Judge #6  

## Architecture Overview

This notebook implements military-grade multi-agent coordination for PNKLN development:

- **Cor (Unified Brain):** Orchestrates agent delegation and decision-making
- **NS (Nervous System):** Agent Mail messaging via GCS buckets
- **JR (Judgment Rule):** Purpose/Reasons/Brakes enforcement per agent
- **YRM (Your Risk Manager):** ATP 5-19 stratification (RA-1 → RA-4)
- **Judge #6:** Gemini-powered code validation with 98% coverage gates

## Prerequisites

```bash
# GCS Buckets (create via Cloud Console or gcloud)
gsutil mb gs://pnkln-agent-mail
gsutil mb gs://pnkln-aiyoujr-logs
gsutil mb gs://pnkln-task-artifacts

# Service Account Permissions
# Grant 'Storage Object Admin' on above buckets
# Grant 'Vertex AI User' for Gemini access
```

## Usage Pattern

1. **Cell 1-6:** Run once per session (environment setup)
2. **Cell 7:** Configure agents for your specific task
3. **Cell 8-9:** Initiate coordination phase
4. **Cell 10:** Execute parallel work with governance
5. **Cell 11:** Review Agent Mail logs + validation results

**Context Management:** Agents auto-pause at 80% context, consolidate notes, and resume.

In [ ]:
# CELL 1: Environment Setup
# Install dependencies (run once per notebook instance)

!pip install -q pyautogen google-cloud-storage google-cloud-aiplatform vertexai

import contextlib
import json
import os
from datetime import datetime

import autogen
import vertexai
from google.cloud import storage
from vertexai.generative_models import GenerativeModel

print("✅ Dependencies loaded")

In [ ]:
# CELL 2: Vertex AI + GCS Initialization

# TODO: Set your GCP project details
PROJECT_ID = "your-gcp-project-id"  # ← CHANGE THIS
LOCATION = "us-central1"
AGENT_MAIL_BUCKET = "pnkln-agent-mail"  # ← CHANGE if using different bucket
GOVERNANCE_BUCKET = "pnkln-aiyoujr-logs"
ARTIFACTS_BUCKET = "pnkln-task-artifacts"

# Initialize Vertex AI
vertexai.init(project=PROJECT_ID, location=LOCATION)

# Initialize GCS client
storage_client = storage.Client(project=PROJECT_ID)
agent_mail_bucket = storage_client.bucket(AGENT_MAIL_BUCKET)
governance_bucket = storage_client.bucket(GOVERNANCE_BUCKET)
artifacts_bucket = storage_client.bucket(ARTIFACTS_BUCKET)

print(f"✅ Vertex AI initialized: {PROJECT_ID} @ {LOCATION}")
print(f"✅ GCS buckets connected: {AGENT_MAIL_BUCKET}, {GOVERNANCE_BUCKET}, {ARTIFACTS_BUCKET}")

In [ ]:
# CELL 3: AiYouJR Governance Layer


class AiYouJRGovernance:
    """Purpose/Reasons/Brakes enforcement + ATP 5-19 risk management + Judge #6 validation"""

    def __init__(self, model_name: str = "gemini-2.0-flash-exp"):
        self.judge_6 = GenerativeModel(model_name)

        # ATP 5-19 Risk Assessment Framework
        self.atp_5_19_thresholds = {
            "RA-1": {
                "description": "Extremely low risk - routine operations",
                "auto_approve": True,
                "coverage_req": 0.95,
                "peer_review": 0,
                "human_gate": False,
            },
            "RA-2": {
                "description": "Low risk - standard development tasks",
                "auto_approve": True,
                "coverage_req": 0.98,
                "peer_review": 1,
                "human_gate": False,
            },
            "RA-3": {
                "description": "Medium risk - critical path, data handling",
                "auto_approve": False,
                "coverage_req": 0.98,
                "peer_review": 2,
                "human_gate": False,
            },
            "RA-4": {
                "description": "High risk - security, compliance, infrastructure",
                "auto_approve": False,
                "coverage_req": 0.99,
                "peer_review": 3,
                "human_gate": True,  # ← Erik approval required
            },
        }

    def assess_task_risk(self, task_description: str, context: dict = None) -> dict:
        """ATP 5-19 stratification using Judge #6"""

        prompt = f"""You are Judge #6, enforcing ATP 5-19 risk assessment for PNKLN operations.

Analyze this task and assign a risk level:

TASK: {task_description}

CONTEXT: {json.dumps(context or {}, indent=2)}

RISK LEVELS:
- RA-1: Routine operations (docs updates, simple refactors)
- RA-2: Standard development (new features, API endpoints)
- RA-3: Critical path (data schemas, auth flows, performance)
- RA-4: High stakes (prod deployments, security, compliance, infrastructure)

Return ONLY valid JSON (no markdown):
{{
  "risk_level": "RA-1|RA-2|RA-3|RA-4",
  "rationale": "Brief explanation",
  "mitigations": ["Required safeguard 1", "Required safeguard 2"],
  "estimated_complexity": "low|medium|high"
}}
"""

        response = self.judge_6.generate_content(prompt)
        result = json.loads(response.text)

        # Attach threshold requirements
        risk_level = result["risk_level"]
        result["requirements"] = self.atp_5_19_thresholds[risk_level]

        # Log assessment
        timestamp = datetime.now().isoformat()
        governance_bucket.blob(
            f"risk_assessments/{timestamp}_{risk_level}.json"
        ).upload_from_string(
            json.dumps({"task": task_description, "assessment": result, "timestamp": timestamp})
        )

        return result

    def validate_completion(
        self, task_id: str, code: str, tests: str, coverage_pct: float, risk_level: str
    ) -> dict:
        """Judge #6 enforcement before task closure"""

        required_coverage = self.atp_5_19_thresholds[risk_level]["coverage_req"]

        if coverage_pct < required_coverage:
            return {
                "approved": False,
                "brake_reason": f"Coverage {coverage_pct:.1%} below {required_coverage:.0%} threshold for {risk_level}",
                "required_action": "Add tests to meet coverage requirement",
            }

        prompt = f"""You are Judge #6, performing final validation for task completion.

TASK ID: {task_id}
RISK LEVEL: {risk_level}

CODE:
```
{code}
```

TESTS:
```
{tests}
```

COVERAGE: {coverage_pct:.1%}

Check for:
1. Security vulnerabilities (SQL injection, XSS, auth bypass)
2. Edge cases not covered by tests
3. Performance anti-patterns (N+1 queries, blocking I/O)
4. Docstring quality (missing params, unclear purpose)
5. Error handling gaps

Return ONLY valid JSON (no markdown):
{{
  "approved": true|false,
  "issues": ["Critical issue 1", ...],
  "warnings": ["Non-blocking concern 1", ...],
  "suggestions": ["Optional improvement 1", ...]
}}
"""

        response = self.judge_6.generate_content(prompt)
        result = json.loads(response.text)

        # Log validation
        timestamp = datetime.now().isoformat()
        governance_bucket.blob(f"validations/{task_id}_{timestamp}.json").upload_from_string(
            json.dumps(
                {
                    "task_id": task_id,
                    "risk_level": risk_level,
                    "coverage": coverage_pct,
                    "validation": result,
                    "timestamp": timestamp,
                }
            )
        )

        return result

    def enforce_brake(self, agent_name: str, brake_condition: str, context: dict) -> None:
        """Log brake activation for audit trail"""
        timestamp = datetime.now().isoformat()
        governance_bucket.blob(f"brakes/{agent_name}_{timestamp}.json").upload_from_string(
            json.dumps(
                {
                    "agent": agent_name,
                    "brake_condition": brake_condition,
                    "context": context,
                    "timestamp": timestamp,
                }
            )
        )
        print(f"🚨 BRAKE ACTIVATED: {agent_name} - {brake_condition}")


# Instantiate governance layer
governance = AiYouJRGovernance()
print("✅ AiYouJR governance layer initialized")

In [ ]:
# CELL 4: Agent Mail System (NS - Nervous System)


class AgentMail:
    """GCS-backed messaging system for inter-agent coordination"""

    def __init__(self, agent_name: str):
        self.agent_name = agent_name
        self.inbox_path = f"inbox/{agent_name}/"
        self.outbox_path = f"outbox/{agent_name}/"

    def send(self, to: str, subject: str, body: str, attachments: list[str] = None) -> str:
        """Send message to another agent or broadcast to all"""

        msg_id = f"{datetime.now().timestamp()}_{self.agent_name}_to_{to}"
        msg = {
            "id": msg_id,
            "from": self.agent_name,
            "to": to,
            "subject": subject,
            "body": body,
            "attachments": attachments or [],
            "timestamp": datetime.now().isoformat(),
            "read": False,
        }

        # Save to outbox
        agent_mail_bucket.blob(f"{self.outbox_path}{msg_id}.json").upload_from_string(
            json.dumps(msg, indent=2)
        )

        # Deliver to recipient inbox (or broadcast to all)
        if to == "*all*":
            # Broadcast to all agent inboxes
            for blob in agent_mail_bucket.list_blobs(prefix="inbox/", delimiter="/"):
                recipient = blob.name.split("/")[1]
                if recipient != self.agent_name:  # Don't send to self
                    agent_mail_bucket.blob(f"inbox/{recipient}/{msg_id}.json").upload_from_string(
                        json.dumps(msg, indent=2)
                    )
        else:
            agent_mail_bucket.blob(f"inbox/{to}/{msg_id}.json").upload_from_string(
                json.dumps(msg, indent=2)
            )

        print(f"📧 {self.agent_name} → {to}: {subject}")
        return msg_id

    def check_inbox(self, unread_only: bool = True, limit: int = 50) -> list[dict]:
        """Retrieve messages from inbox"""

        blobs = list(agent_mail_bucket.list_blobs(prefix=self.inbox_path, max_results=limit))
        messages = []

        for blob in blobs:
            try:
                msg = json.loads(blob.download_as_text())
                if unread_only and msg.get("read", False):
                    continue
                messages.append(msg)
            except Exception as e:
                print(f"⚠️  Error reading {blob.name}: {e}")

        messages.sort(key=lambda x: x["timestamp"])
        return messages

    def mark_read(self, msg_id: str) -> None:
        """Mark message as read"""
        blob_name = f"{self.inbox_path}{msg_id}.json"
        blob = agent_mail_bucket.blob(blob_name)

        if blob.exists():
            msg = json.loads(blob.download_as_text())
            msg["read"] = True
            blob.upload_from_string(json.dumps(msg, indent=2))

    def get_thread(self, subject_filter: str) -> list[dict]:
        """Get all messages in a thread by subject"""
        all_messages = self.check_inbox(unread_only=False, limit=200)
        return [msg for msg in all_messages if subject_filter.lower() in msg["subject"].lower()]


print("✅ Agent Mail system initialized")

In [ ]:
# CELL 5: Task Coordination Utilities


class TaskCoordinator:
    """Manages task distribution and progress tracking"""

    def __init__(self, plan_file_path: str):
        self.plan_file = plan_file_path
        self.tasks = self._parse_plan()

    def _parse_plan(self) -> list[dict]:
        """Extract tasks from PLAN.md file"""
        # TODO: Implement based on your PLAN.md format
        # This is a placeholder - adapt to your actual plan structure
        return []

    def claim_task(self, task_id: str, agent_name: str) -> bool:
        """Agent claims a task"""
        # TODO: Implement atomic task claiming with GCS object locking
        pass

    def update_progress(self, task_id: str, progress_pct: int, notes: str) -> None:
        """Update task progress (25/50/75/100% checkpoints)"""
        timestamp = datetime.now().isoformat()
        artifacts_bucket.blob(f"progress/{task_id}_{timestamp}.json").upload_from_string(
            json.dumps(
                {
                    "task_id": task_id,
                    "progress": progress_pct,
                    "notes": notes,
                    "timestamp": timestamp,
                }
            )
        )

    def get_remaining_tasks(self, risk_level_filter: str | None = None) -> list[dict]:
        """Get unclaimed tasks, optionally filtered by risk level"""
        # TODO: Implement based on your task tracking mechanism
        pass


# TODO: Set path to your PLAN.md file
# coordinator = TaskCoordinator("/path/to/PLAN.md")

print("✅ Task coordination utilities loaded")

In [ ]:
# CELL 6: Context Management Helper


class ContextManager:
    """Monitors token usage and triggers consolidation at 80% threshold"""

    def __init__(self, max_tokens: int = 100000):
        self.max_tokens = max_tokens
        self.current_tokens = 0
        self.consolidation_threshold = 0.8

    def update_usage(self, tokens: int) -> None:
        self.current_tokens += tokens

    def needs_consolidation(self) -> bool:
        return self.current_tokens >= (self.max_tokens * self.consolidation_threshold)

    def consolidate(self, agent_mail: AgentMail, summary: str) -> None:
        """Trigger consolidation: save summary, reset counter"""
        timestamp = datetime.now().isoformat()
        artifacts_bucket.blob(
            f"consolidations/{agent_mail.agent_name}_{timestamp}.txt"
        ).upload_from_string(summary)

        agent_mail.send(
            to="cor_orchestrator",
            subject="Context Consolidation",
            body=f"Reached 80% context limit. Summary saved to GCS. Resetting and resuming.\n\n{summary[:500]}...",
        )

        self.current_tokens = 0
        print(f"♻️  Context consolidated for {agent_mail.agent_name}")


print("✅ Context management helper loaded")

---
## Agent Configuration

**Customize the agents below for your specific task.**

Each agent requires:
- **name**: Unique identifier
- **role**: Functional specialization
- **purpose**: Primary objective (AiYouJR "P")
- **reasons**: Justifications for actions (AiYouJR "R")
- **brakes**: Non-negotiable constraints (AiYouJR "B")
- **llm_config**: Model and temperature settings

In [ ]:
# CELL 7: Define Agent Configuration

agent_configs = [
    {
        "name": "WhiteCastle",
        "role": "Backend/API Development",
        "purpose": "Implement high-performance backend services with async patterns",
        "reasons": [
            "Performance-critical execution paths",
            "Memory safety for long-running services",
            "API contract enforcement",
        ],
        "brakes": [
            "No unsafe blocks without RA-3 approval + justification",
            "Test coverage >= 98% before task completion",
            "No external dependencies without security audit",
            "All API endpoints require authentication",
        ],
        "llm_config": {
            "model": "gemini-2.0-flash-exp",
            "temperature": 0.2,  # Low temp for deterministic code
        },
    },
    {
        "name": "BrownSnow",
        "role": "DevOps/Infrastructure",
        "purpose": "Cloud infrastructure automation and CI/CD pipeline management",
        "reasons": [
            "Deployment reliability and consistency",
            "Hardware coordination for edge compute",
            "Cost optimization through automation",
        ],
        "brakes": [
            "No production deployments without RA-2 validation",
            "Rollback plan required before any infrastructure change",
            "Monitoring/alerting mandatory for all services",
            "Terraform state changes require peer review",
        ],
        "llm_config": {
            "model": "gemini-2.0-flash-exp",
            "temperature": 0.1,  # Very low temp for infrastructure
        },
    },
    {
        "name": "OrangeCreek",
        "role": "Frontend/UX",
        "purpose": "User interfaces with accessibility and performance standards",
        "reasons": [
            "User experience drives adoption",
            "Accessibility compliance (legal + ethical)",
            "Performance impacts conversion rates",
        ],
        "brakes": [
            "WCAG 2.1 AA compliance minimum",
            "Performance budget: Largest Contentful Paint < 3s",
            "Mobile-first design required",
            "No inline styles - use design system tokens",
        ],
        "llm_config": {
            "model": "gemini-2.0-flash-exp",
            "temperature": 0.3,  # Slightly higher for creative UI work
        },
    },
    # ADD MORE AGENTS AS NEEDED
]

print(f"✅ Configured {len(agent_configs)} agents")
for cfg in agent_configs:
    print(f"   - {cfg['name']}: {cfg['role']}")

In [ ]:
# CELL 8: Instantiate AutoGen Agents with AiYouJR Governance

agents = []

for config in agent_configs:
    agent_mail = AgentMail(config["name"])
    context_mgr = ContextManager()

    # Construct system message with PRB (Purpose/Reasons/Brakes)
    system_message = f"""You are {config["name"]}, an autonomous agent specialized in {config["role"]}.

═══════════════════════════════════════
AIYOUJR GOVERNANCE FRAMEWORK
═══════════════════════════════════════

PURPOSE: {config["purpose"]}

REASONS (Your justifications for action):
{chr(10).join(f"  • {r}" for r in config["reasons"])}

BRAKES (Non-negotiable constraints - NEVER violate):
{chr(10).join(f"  🚨 {b}" for b in config["brakes"])}

═══════════════════════════════════════
WORKFLOW PROTOCOL
═══════════════════════════════════════

1. STARTUP SEQUENCE:
   - Check Agent Mail inbox: agent_mail.check_inbox()
   - Send introduction to 'cor_orchestrator'
   - Read coordination documents (AGENTS.md, AIYOUJR_DOCTRINE.md, PLAN.md)

2. TASK ASSESSMENT:
   - For each potential task, call: governance.assess_task_risk(task_description)
   - Review ATP 5-19 risk level (RA-1 through RA-4)
   - Check if peer review required (RA-3+)
   - NEVER claim RA-4 tasks without human approval

3. COORDINATION:
   - Use Agent Mail for all inter-agent communication
   - For RA-3+ tasks: send proposal to peers, wait for 2+ confirmations
   - Update PLAN.md with claimed tasks and ETA

4. EXECUTION:
   - Send progress updates at 25%, 50%, 75%, 100% via Agent Mail
   - If you detect a BRAKE violation, immediately:
     a) Stop work on that task
     b) Call governance.enforce_brake()
     c) Send Agent Mail to cor_orchestrator with details
   - Monitor context usage: if context_mgr.needs_consolidation(), pause and consolidate

5. VALIDATION:
   - Before marking task complete, call governance.validate_completion()
   - If Judge #6 rejects: address issues, resubmit
   - If approved: update PLAN.md, send completion notice via Agent Mail

6. CONTEXT MANAGEMENT:
   - At 80% context: create summary, save to GCS, send to cor_orchestrator
   - Reset and resume from last checkpoint

═══════════════════════════════════════
AVAILABLE TOOLS
═══════════════════════════════════════

- agent_mail.send(to, subject, body): Send messages to other agents
- agent_mail.check_inbox(): Retrieve new messages
- governance.assess_task_risk(task): Get ATP 5-19 risk level
- governance.validate_completion(...): Judge #6 final check
- governance.enforce_brake(...): Log brake activation
- context_mgr.needs_consolidation(): Check if nearing context limit

═══════════════════════════════════════

Remember: Your PURPOSE guides what you do. Your REASONS justify why. Your BRAKES define what you NEVER do.

Execute with military precision. Coordinate with peers. Enforce governance without exception.
"""

    # Create AutoGen agent
    agent = autogen.AssistantAgent(
        name=config["name"],
        system_message=system_message,
        llm_config={
            "config_list": [
                {
                    "model": config["llm_config"]["model"],
                    "api_type": "google",
                    "api_key": os.environ.get("GOOGLE_API_KEY"),  # Or use Vertex AI auth
                    "temperature": config["llm_config"]["temperature"],
                }
            ]
        },
    )

    # Attach PNKLN utilities
    agent.agent_mail = agent_mail
    agent.context_mgr = context_mgr
    agent.governance = governance
    agent.config = config

    agents.append(agent)
    print(f"✅ Initialized: {config['name']}")

print(f"\n✅ All {len(agents)} agents initialized with AiYouJR governance")

In [ ]:
# CELL 9: Create Group Chat with Cor Orchestrator

# Group chat allows all agents to see each other's messages
groupchat = autogen.GroupChat(
    agents=agents,
    messages=[],
    max_round=100,  # Prevents infinite loops
)

# Manager acts as Cor (unified brain)
manager = autogen.GroupChatManager(
    groupchat=groupchat,
    llm_config={
        "config_list": [
            {
                "model": "gemini-2.0-flash-exp",
                "api_type": "google",
                "api_key": os.environ.get("GOOGLE_API_KEY"),
                "temperature": 0,
            }
        ]
    },
    system_message="""You are Cor, the orchestrator for PNKLN multi-agent operations.

Your responsibilities:
- Facilitate agent coordination
- Monitor for brake violations
- Escalate RA-4 tasks to Erik
- Ensure ATP 5-19 compliance
- Intervene if agents deadlock or drift off-task

Stay silent unless coordination is needed. Agents should self-organize.
""",
)

print("✅ Group chat initialized with Cor orchestrator")

In [ ]:
# CELL 10: Initiate Multi-Agent Coordination

# TODO: Customize this prompt for your specific task
initial_coordination_prompt = """═══════════════════════════════════════
COORDINATION PHASE INITIATED
═══════════════════════════════════════

All agents must complete the following sequence:

1. REGISTRATION:
   - Send introduction via Agent Mail to 'cor_orchestrator'
   - Include: your name, role, capabilities, risk tolerance

2. DOCUMENT REVIEW:
   - Read: /home/claude/AGENTS.md
   - Read: /home/claude/AIYOUJR_DOCTRINE.md
   - Read: /home/claude/PLAN_TO_INTEGRATE_BEVY_ENGINE.md  # ← CHANGE THIS

3. TASK IDENTIFICATION:
   - List all remaining incomplete tasks from PLAN.md
   - For each task, run: governance.assess_task_risk()
   - Document risk levels in Agent Mail thread: "Bevy Integration Risk Assessment"  # ← CHANGE THIS

4. WORK DISTRIBUTION:
   - Propose task assignments based on specialization
   - RA-1/RA-2: Self-assign after notifying peers
   - RA-3: Require 2-agent peer review before claiming
   - RA-4: Flag for Erik approval, do not claim
   - Use Agent Mail thread: "Work Distribution - [PROJECT_NAME]" # ← CHANGE THIS

5. WAIT FOR CONSENSUS:
   - All agents must confirm work distribution
   - Resolve conflicts via Agent Mail negotiation
   - Cor will intervene only if deadlock

6. EXECUTION:
   Once consensus reached, proceed with:
   "Execute assigned tasks systematically. Update PLAN.md inline.
    Send Agent Mail at 25/50/75/100% checkpoints. Enforce brakes without exception."

═══════════════════════════════════════
BEGIN COORDINATION
═══════════════════════════════════════
"""

# Start the group chat
print("🚀 Initiating multi-agent coordination...\n")
manager.initiate_chat(manager, message=initial_coordination_prompt)

print("\n✅ Coordination phase complete")

In [ ]:
# CELL 11: Monitor Agent Mail & Governance Logs


def review_agent_mail(agent_name: str = None, limit: int = 20):
    """Review Agent Mail messages"""
    if agent_name:
        mail = AgentMail(agent_name)
        messages = mail.check_inbox(unread_only=False, limit=limit)
        print(f"\n📧 {agent_name} INBOX ({len(messages)} messages):\n")
    else:
        print("\n📧 ALL AGENT MAIL:\n")
        messages = []
        for blob in agent_mail_bucket.list_blobs(prefix="inbox/"):
            with contextlib.suppress(BaseException):
                messages.append(json.loads(blob.download_as_text()))
        messages = sorted(messages, key=lambda x: x["timestamp"])[-limit:]

    for msg in messages:
        print(f"[{msg['timestamp']}] {msg['from']} → {msg['to']}")
        print(f"  Subject: {msg['subject']}")
        print(
            f"  Body: {msg['body'][:200]}..."
            if len(msg["body"]) > 200
            else f"  Body: {msg['body']}"
        )
        print()


def review_governance_logs(log_type: str = "all", limit: int = 10):
    """Review AiYouJR governance logs"""
    prefixes = {"risk": "risk_assessments/", "validations": "validations/", "brakes": "brakes/"}

    if log_type == "all":
        for name, prefix in prefixes.items():
            print(f"\n🛡️  {name.upper()}:")
            blobs = list(governance_bucket.list_blobs(prefix=prefix, max_results=limit))
            for blob in blobs[-5:]:  # Last 5 of each type
                print(f"  - {blob.name}")
    else:
        prefix = prefixes.get(log_type)
        if not prefix:
            print(f"Invalid log_type: {log_type}. Use 'risk', 'validations', 'brakes', or 'all'")
            return

        print(f"\n🛡️  {log_type.upper()}:")
        blobs = list(governance_bucket.list_blobs(prefix=prefix, max_results=limit))
        for blob in blobs:
            log = json.loads(blob.download_as_text())
            print(json.dumps(log, indent=2))
            print()


# Example usage:
# review_agent_mail("WhiteCastle")
# review_agent_mail()  # All agents
# review_governance_logs("brakes")
# review_governance_logs()  # All logs

print("✅ Monitoring utilities loaded")
print("\nUsage:")
print("  review_agent_mail('WhiteCastle')  # Single agent inbox")
print("  review_agent_mail()                # All agent mail")
print("  review_governance_logs('brakes')   # Brake violations only")
print("  review_governance_logs()           # All governance logs")

---
## Post-Execution Analysis

After agents complete their work, use the cells below to analyze results.

In [ ]:
# CELL 12: Generate Execution Report


def generate_execution_report():
    """Compile comprehensive report of agent activities"""

    report = {
        "timestamp": datetime.now().isoformat(),
        "agents": [],
        "risk_assessments": [],
        "validations": [],
        "brake_activations": [],
        "message_count": 0,
    }

    # Gather agent summaries
    for agent_cfg in agent_configs:
        mail = AgentMail(agent_cfg["name"])
        inbox = mail.check_inbox(unread_only=False)
        outbox_blobs = list(agent_mail_bucket.list_blobs(prefix=f"outbox/{agent_cfg['name']}/"))

        report["agents"].append(
            {
                "name": agent_cfg["name"],
                "role": agent_cfg["role"],
                "messages_received": len(inbox),
                "messages_sent": len(outbox_blobs),
            }
        )
        report["message_count"] += len(inbox) + len(outbox_blobs)

    # Gather governance logs
    for blob in governance_bucket.list_blobs(prefix="risk_assessments/"):
        report["risk_assessments"].append(json.loads(blob.download_as_text()))

    for blob in governance_bucket.list_blobs(prefix="validations/"):
        report["validations"].append(json.loads(blob.download_as_text()))

    for blob in governance_bucket.list_blobs(prefix="brakes/"):
        report["brake_activations"].append(json.loads(blob.download_as_text()))

    # Save report
    report_filename = f"execution_report_{datetime.now().strftime('%Y%m%d_%H%M%S')}.json"
    artifacts_bucket.blob(f"reports/{report_filename}").upload_from_string(
        json.dumps(report, indent=2)
    )

    print("\n" + "=" * 60)
    print("EXECUTION REPORT")
    print("=" * 60)
    print(f"\nAgents: {len(report['agents'])}")
    print(f"Total Messages: {report['message_count']}")
    print(f"Risk Assessments: {len(report['risk_assessments'])}")
    print(f"Validations: {len(report['validations'])}")
    print(f"🚨 Brake Activations: {len(report['brake_activations'])}")

    if report["brake_activations"]:
        print("\n⚠️  BRAKE VIOLATIONS DETECTED:")
        for brake in report["brake_activations"]:
            print(f"  - {brake['agent']}: {brake['brake_condition']}")

    print(f"\n📊 Full report saved: gs://{ARTIFACTS_BUCKET}/reports/{report_filename}")
    print("=" * 60)

    return report


# Run report
# report = generate_execution_report()

print("✅ Report generator loaded. Run: generate_execution_report()")

---
## Template Complete

**Next Steps:**

1. ✅ Customize Cell 7 with your agent configurations
2. ✅ Update Cell 10 with your specific task/plan
3. ✅ Run Cells 1-9 to initialize agents
4. ✅ Execute Cell 10 to start coordination
5. ✅ Monitor via Cell 11 (Agent Mail + governance logs)
6. ✅ Generate report via Cell 12

**PNKLN Core Stack™ - Bootstrap from $0K to $1.33T**  
*Military-grade execution. Zero compromise on governance.*